In [28]:
import numpy as np
import pandas as pd

In [29]:
df = pd.read_csv("fluencybank_labels.csv")

In [30]:
df

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,WordRep,DifficultToUnderstand,Interjection,NoStutteredWords,NaturalPause,Music,NoSpeech
0,FluencyBank,10,0,88960,136960,0,0,0,1,0,0,0,0,2,0,0,0
1,FluencyBank,10,1,1271520,1319520,0,0,0,0,0,0,0,0,3,0,0,0
2,FluencyBank,10,2,1813760,1861760,0,0,1,0,0,0,0,0,2,0,0,0
3,FluencyBank,10,3,1842720,1890720,0,0,1,0,0,0,0,0,2,1,0,0
4,FluencyBank,10,4,1893280,1941280,0,0,0,0,0,3,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4139,FluencyBank,985,191,13483520,13531520,0,0,0,1,0,0,0,0,2,0,0,0
4140,FluencyBank,985,192,13517280,13565280,0,0,0,2,1,0,0,0,0,1,0,0
4141,FluencyBank,985,193,13568960,13616960,0,0,0,2,2,1,0,0,0,1,0,0
4142,FluencyBank,985,194,1683520,1731520,0,0,0,0,3,0,0,1,0,0,0,0


In [31]:
df['EpId'] = df['EpId'].astype(str).str.zfill(3)

In [32]:
df['Clip'] = (
    df['Show'].astype(str) + "_" +
    df['EpId'].astype(str) + "_" +
    df['ClipId'].astype(str)
)
df

,Show,EpId,ClipId,Start,Stop,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,WordRep,DifficultToUnderstand,Interjection,NoStutteredWords,NaturalPause,Music,NoSpeech,Clip
0,FluencyBank,010,0,88960,136960,0,0,0,1,0,0,0,0,2,0,0,0,FluencyBank_010_0
1,FluencyBank,010,1,1271520,1319520,0,0,0,0,0,0,0,0,3,0,0,0,FluencyBank_010_1
2,FluencyBank,010,2,1813760,1861760,0,0,1,0,0,0,0,0,2,0,0,0,FluencyBank_010_2
3,FluencyBank,010,3,1842720,1890720,0,0,1,0,0,0,0,0,2,1,0,0,FluencyBank_010_3
4,FluencyBank,010,4,1893280,1941280,0,0,0,0,0,3,0,0,0,0,0,0,FluencyBank_010_4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4139,FluencyBank,985,191,13483520,13531520,0,0,0,1,0,0,0,0,2,0,0,0,FluencyBank_985_191
4140,FluencyBank,985,192,13517280,13565280,0,0,0,2,1,0,0,0,0,1,0,0,FluencyBank_985_192
4141,FluencyBank,985,193,13568960,13616960,0,0,0,2,2,1,0,0,0,1,0,0,FluencyBank_985_193
4142,FluencyBank,985,194,1683520,1731520,0,0,0,0,3,0,0,1,0,0,0,0,FluencyBank_985_194


In [33]:
df['audio_path'] = (
    "fluencybank_CLIP/fluencybank_CLIP/FluencyBank/" +
    df['EpId'].astype(str) + "/" +
    df['Clip'] + ".wav"
)

In [34]:
df = df.drop(columns=['Start','Stop','DifficultToUnderstand','NaturalPause','Music'])

In [35]:
print(df['audio_path'])

0       fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
1       fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
2       fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
3       fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
4       fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
                              ...                        
4139    fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
4140    fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
4141    fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
4142    fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
4143    fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
Name: audio_path, Length: 4144, dtype: str


In [36]:
import os
test_path = df['audio_path'].iloc[0]
print(f"🔍 Checking first file at: {os.path.abspath(test_path)}")
if not os.path.exists(test_path):
    print("❌ ERROR: File STILL not found! Please stop and double-check your folder structure.")
else:
    print("✅ File found! Pathing is correct.\n")

🔍 Checking first file at: c:\MyStuff\SJSU\Spring 2026\Ling 230\DisfluencyProject\disfluency_detection\ksh\stl\fluencybank_CLIP\fluencybank_CLIP\FluencyBank\010\FluencyBank_010_0.wav
✅ File found! Pathing is correct.



In [37]:
import pandas as pd
import torch
import librosa
import os
from transformers import WhisperFeatureExtractor, WhisperModel
from tqdm.notebook import tqdm # visualization
import numpy as np 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader # for training
from sklearn.utils.class_weight import compute_class_weight # class weights
from sklearn.model_selection import train_test_split # split data
from sklearn.preprocessing import LabelEncoder #transform labels
import matplotlib.pyplot as plt # visualization
import seaborn as sns # visualization
from sklearn.metrics import confusion_matrix, classification_report # visualization
import copy # copy and save model 
import math
import gc

In [38]:
# Set up the Whisper Model
model_name = "openai/whisper-base"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name)

# find Mac GPU, also allowed for NVIDIA GPU
if torch.backends.mps.is_available():
    device = "mps"
    print("Using Mac GPU (MPS) for acceleration!")
elif torch.cuda.is_available():
    device = "cuda"
    print("Using NVIDIA GPU (CUDA) for acceleration!")
else:
    device = "cpu"
    print("Using CPU. (No GPU detected)")

model.to(device)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using NVIDIA GPU (CUDA) for acceleration!


WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 512)
    (layers): ModuleList(
      (0-5): 6 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=False)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=512, out_features=2048, bias=True)
        (fc2): Linear(in_features=2048, out_features=512, bias=True)
        (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )


In [39]:
def get_whisper_embedding_sequence(audio_path):
    if not os.path.exists(audio_path): return None
    try:
        # Load audio at 16k Hz
        speech_array, sampling_rate = librosa.load(audio_path, sr=16000)
        inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)
        
        with torch.no_grad():
            encoder_outputs = model.encoder(input_features)
            
        # remove squeeze dimension
        embedding = encoder_outputs.last_hidden_state.squeeze(0).cpu().numpy()
        embedding = embedding[:150]

        return embedding 
    except Exception as e:
        print(f"Error: {e}")
        return None

In [40]:
import numpy as np
from tqdm import tqdm

TARGET_T = 150
TARGET_D = 512  # change to 768 if whisper-small
N = len(df)

print("Creating memory-mapped .npy file...")

# Create final .npy file directly
embeddings = np.lib.format.open_memmap(
    "fb_embeddings.npy",
    mode="w+",
    dtype=np.float32,
    shape=(N, TARGET_T, TARGET_D),
)

valid_mask = np.zeros(N, dtype=bool)

print("Processing audio files...")

for idx, path in tqdm(enumerate(df["audio_path"]), total=N):
    emb = get_whisper_embedding_sequence(path)

    if emb is None or emb.shape != (TARGET_T, TARGET_D):
        continue

    embeddings[idx] = emb
    valid_mask[idx] = True

# Flush everything to disk
del embeddings  # important — ensures file is written

print("Initial save complete.")

Creating memory-mapped .npy file...
Processing audio files...


100%|██████████| 4144/4144 [01:01<00:00, 67.39it/s]


Initial save complete.


In [41]:
import numpy as np
import os

valid_indices = np.where(valid_mask)[0]

in_path = "fb_embeddings.npy"
out_path = "fb_embeddings_compact.npy"

# Load memory-mapped file (read-only)
full = np.load(in_path, mmap_mode="r")

# Create compact final file (NEW NAME)
compact = np.lib.format.open_memmap(
    out_path,
    mode="w+",
    dtype=np.float32,
    shape=(len(valid_indices), TARGET_T, TARGET_D),
)

for i, idx in enumerate(valid_indices):
    compact[i] = full[idx]

# IMPORTANT: release file handles on Windows
del compact
del full

clean_df = df.iloc[valid_indices].copy()
clean_df.to_csv("fb_with_paths.csv", index=False)

print("Final compact file saved:", out_path)
print("Final shape:", (len(valid_indices), TARGET_T, TARGET_D))

# Optional: replace original AFTER both are closed
os.replace(out_path, in_path)

Final compact file saved: fb_embeddings_compact.npy
Final shape: (4144, 150, 512)


In [42]:
# Cnn + transformer

class TemporalCNNEncoder(nn.Module):
    def __init__(self):
        super(TemporalCNNEncoder, self).__init__()
        # Whisper base dim is 512. We treat these as "Channels" for the CNN
        self.conv1 = nn.Conv1d(in_channels=512, out_channels=256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(256)
        self.conv2 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu = nn.GELU()
        self.pool = nn.MaxPool1d(kernel_size=2) 

    def forward(self, x):
        # x enters as [Batch, 150, 512]. Conv1d: [Batch, Channels, Time]
        x = x.transpose(1, 2) 
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        return x.transpose(1, 2) # Returns [Batch, 75, 128] for Transformer
    

class TransformerBrain(nn.Module):
    def __init__(self, num_labels, d_model=128, seq_length=75):
        super(TransformerBrain, self).__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_length, d_model) * 0.01)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=512, dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc = nn.Linear(d_model, num_labels)

    def forward(self, x):
        x = x + self.pos_embedding
        x = self.transformer(x)
        x = x.mean(dim=1) 
        return self.fc(x)
    

class EndToEndStutterModel(nn.Module):
    def __init__(self, num_labels):
        super(EndToEndStutterModel, self).__init__()
        self.cnn = TemporalCNNEncoder()
        self.transformer = TransformerBrain(num_labels)

    def forward(self, x):
        return self.transformer(self.cnn(x))

In [43]:
label_cols = ['Block', 'Prolongation', 'SoundRep', 'WordRep', 'Interjection']
num_labels = len(label_cols)
model = EndToEndStutterModel(num_labels=num_labels).to(device)

In [44]:
model.load_state_dict(torch.load("downsampled_model2k.pt", weights_only=True))

<All keys matched successfully>

In [47]:
label_cols = ['Block', 'Prolongation', 'SoundRep', 'WordRep', 'Interjection'] 
print("Loading data...")
X_raw = np.load('fb_embeddings.npy') 

# Slice just to make sure 150 embedded dimension. 
X = X_raw[:, :150, :] 

# Free up the old memory immediately
print(f"Original Shape: {X_raw.shape} | Optimized Shape: {X.shape}")
del X_raw
gc.collect() 

df = pd.read_csv('fb_with_paths.csv')

y = df[label_cols].to_numpy(dtype=np.float32)


X_test_tensor = torch.tensor(X, dtype=torch.float32)
y_test_tensor = torch.tensor(y, dtype=torch.float32)


test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)


Loading data...
Original Shape: (4144, 150, 512) | Optimized Shape: (4144, 150, 512)


In [48]:
df.head()

,Show,EpId,ClipId,Unsure,PoorAudioQuality,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,NoSpeech,Clip,audio_path
0,FluencyBank,10,0,0,0,0,1,0,0,0,2,0,FluencyBank_010_0,fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
1,FluencyBank,10,1,0,0,0,0,0,0,0,3,0,FluencyBank_010_1,fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
2,FluencyBank,10,2,0,0,1,0,0,0,0,2,0,FluencyBank_010_2,fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
3,FluencyBank,10,3,0,0,1,0,0,0,0,2,0,FluencyBank_010_3,fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...
4,FluencyBank,10,4,0,0,0,0,0,3,0,0,0,FluencyBank_010_4,fluencybank_CLIP/fluencybank_CLIP/FluencyBank/...


In [49]:
# Evaluate
model.eval()
all_probs = []


with torch.no_grad():
    for batch_X, _ in test_loader:
        batch_X = batch_X.to(device)
        probs = torch.sigmoid(model(batch_X))
        all_probs.append(probs.cpu())

probs_np = torch.cat(all_probs, dim=0).numpy()

In [50]:

probs_df = pd.DataFrame(probs_np, columns=label_cols)
probs_df.insert(0, "Clip", df["Clip"])
len(label_cols) == probs_np.shape[1]
len(df.Clip) == probs_np.shape[0]
#probs_df.to_csv("uclass-testprob_multilabel.csv", index=False)

True

In [51]:
probs_df.to_csv("fb-testprob_multilabel.csv", index=False)